# Ecuador Real Estate Rental Price Prediction - EDA

## Exploratory Data Analysis

This notebook covers:
1. Data loading and exploration
2. Data cleaning and normalization
3. Handling missing values
4. Descriptive and statistical analysis
5. Feature engineering and price categorization

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Section 1: Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('data/real_state_ecuador_dataset.csv')

print("Dataset Shape:", df.shape)
print("\n" + "="*80)
print("Column Names and Data Types:")
print("="*80)
print(df.dtypes)
print("\n" + "="*80)
print("First few rows:")
print("="*80)
df.head()

## Section 2: Data Cleaning and Normalization

In [ ]:
# Clean Lugar column - extract the main location
def normalize_lugar(lugar_str):
    if pd.isna(lugar_str):
        return np.nan
    
    lugar_str = str(lugar_str).strip()
    parts = [p.strip() for p in lugar_str.split(',')]
    
    if len(parts) > 1:
        return parts[1]
    else:
        return parts[0]

df['Lugar_Normalizado'] = df['Lugar'].apply(normalize_lugar)

print("Sample normalized locations:")
print(df[['Lugar', 'Lugar_Normalizado']].drop_duplicates().head(15))

## Section 3: Handle Missing Values

In [ ]:
# Check missing values
print("Missing values by column:")
print(df.isnull().sum())
print("\nMissing values percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

# Handle missing values
df_clean = df.dropna(subset=['Precio', 'Lugar_Normalizado', 'Area']).copy()

# Convert numeric columns
numeric_cols = ['Num. dormitorios', 'Num. banos', 'Num. garages', 'Area', 'Precio']
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Fill missing values with median by Provincia
for col in ['Num. dormitorios', 'Num. banos', 'Num. garages']:
    df_clean[col] = df_clean.groupby('Provincia')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Fill any remaining NaN with overall median
df_clean[['Num. dormitorios', 'Num. banos', 'Num. garages']] = df_clean[
    ['Num. dormitorios', 'Num. banos', 'Num. garages']
].fillna(df_clean[['Num. dormitorios', 'Num. banos', 'Num. garages']].median())

print(f"\nDataset shape after cleaning: {df_clean.shape}")
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

## Section 4: Descriptive Analysis

In [ ]:
# Total properties
print(f"Total properties in dataset: {len(df_clean)}")

# Properties by Provincia
print("\nProperties by Provincia:")
print(df_clean['Provincia'].value_counts().sort_values(ascending=False))

# Price statistics
print("\n" + "="*50)
print("Price Statistics:")
print("="*50)
print(f"Mean Price: ${df_clean['Precio'].mean():.2f}")
print(f"Median Price: ${df_clean['Precio'].median():.2f}")
print(f"Std Dev: ${df_clean['Precio'].std():.2f}")
print(f"Min Price: ${df_clean['Precio'].min():.2f}")
print(f"Max Price: ${df_clean['Precio'].max():.2f}")

# Top locations
print("\nTop 10 locations by property count:")
print(df_clean['Lugar_Normalizado'].value_counts().head(10))

# Price by location (top 10)
top_lugares = df_clean['Lugar_Normalizado'].value_counts().head(10).index
precio_by_lugar = df_clean[df_clean['Lugar_Normalizado'].isin(top_lugares)].groupby(
    'Lugar_Normalizado'
)['Precio'].agg(['count', 'mean', 'median']).round(2)
print("\nPrice statistics by top locations:")
print(precio_by_lugar)

## Section 5: Create Price Category Column

In [ ]:
# Create price categories based on quartiles per location
def categorize_price_by_lugar(df):
    df_copy = df.copy()
    df_copy['Tipo_Precio'] = 'Medio'
    
    for loc in df_copy['Lugar_Normalizado'].unique():
        loc_mask = df_copy['Lugar_Normalizado'] == loc
        q1 = df_copy[loc_mask]['Precio'].quantile(0.25)
        q3 = df_copy[loc_mask]['Precio'].quantile(0.75)
        
        df_copy.loc[(loc_mask) & (df_copy['Precio'] < q1), 'Tipo_Precio'] = 'Económico'
        df_copy.loc[(loc_mask) & (df_copy['Precio'] > q3), 'Tipo_Precio'] = 'Lujo'
    
    return df_copy

df_clean = categorize_price_by_lugar(df_clean)

print("Price Category Distribution:")
print(df_clean['Tipo_Precio'].value_counts())
print(f"\nPercentage distribution:")
print((df_clean['Tipo_Precio'].value_counts() / len(df_clean) * 100).round(2))

# Save cleaned data
df_clean.to_csv('data/real_state_clean.csv', index=False)
print("\n✓ Cleaned dataset saved to 'data/real_state_clean.csv'")